# Data Preprocessing

# Data Preprocessing
This notebook loads raw 15-minute electricity demand data and creates cleaned_data.csv.

## Steps:
1. Load raw dataset from data/raw/synthetic_realistic_load.csv
2. Detect timestamp and demand columns automatically
3. Convert timestamp to datetime and sort
4. Remove duplicate timestamps
5. Remove impossible values (missing or negative demand)
6. Remove extreme spikes above median + 10 × IQR
7. Fill gaps using linear interpolation
8. Remove outliers outside Q1 − 3 × IQR … Q3 + 3 × IQR
9. Resample to 15-minute intervals (mean) and interpolate the removed readings
10. Add time features (Date, Hour, Minute, DayOfWeek, Month, IsWeekend)
11. Save cleaned data to notebooks/outputs/cleaned_data.csv

Steps 5–9 use the same rules, in the same order, as `backend/app/ml/preprocessing.py`.

> **Note (19 Oct 2026): this notebook saves its files to `notebooks/outputs/`, not `data/processed/` or `models/`.**
>
> The dashboard and API read `data/processed/` and `models/`. So that running this notebook can never overwrite those production files, every file it saves goes to `notebooks/outputs/` instead (models to `notebooks/outputs/models/`). Notebooks 01 → 05 read each other's files from that folder, so the chain still works end to end.
>
> The logic is the same as the app modules in `backend/app/ml/`. The last cell compares this notebook's results with the production files (read-only).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Set up paths
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Outputs go to notebooks/outputs/ so production data/processed/ is never overwritten
DATA_PROCESSED_DIR = PROJECT_ROOT / "notebooks" / "outputs"
RAW_DATA_PATH = DATA_RAW_DIR / "synthetic_realistic_load.csv"
CLEANED_DATA_PATH = DATA_PROCESSED_DIR / "cleaned_data.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Processed data path: {DATA_PROCESSED_DIR}")

Project root: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence
Raw data path: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\data\raw\synthetic_realistic_load.csv
Processed data path: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs


In [2]:

# Load raw dataset
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found at {RAW_DATA_PATH}")

print(f"Loading raw dataset from: {RAW_DATA_PATH}")
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Raw dataset loaded successfully!")
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"First few rows:")
print(df_raw.head())


Loading raw dataset from: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\data\raw\synthetic_realistic_load.csv
Raw dataset loaded successfully!
Shape: (189793, 4)
Columns: ['Timestamp', 'Load Demand (kW)', 'Temperature (°C)', 'Solar Irradiance (W/m²)']
First few rows:
             Timestamp  Load Demand (kW)  Temperature (°C)  \
0  2020-01-01 00:00:00           1554.47             22.51   
1  2020-01-01 00:15:00           1471.91             22.49   
2  2020-01-01 00:30:00           1581.97             22.02   
3  2020-01-01 00:45:00           1593.76             22.90   
4  2020-01-01 01:00:00           1416.25             22.67   

   Solar Irradiance (W/m²)  
0                      0.0  
1                      0.0  
2                      0.0  
3                      0.0  
4                      0.0  


In [3]:
# Detect timestamp column automatically
timestamp_cols = [col for col in df_raw.columns if any(keyword in col.lower() for keyword in ['time', 'date', 'timestamp'])]
if timestamp_cols:
    ts_col = timestamp_cols[0]
else:
    for col in df_raw.columns:
        try:
            pd.to_datetime(df_raw[col].head(100))
            ts_col = col
            break
        except:
            continue

# Detect demand column — prefer columns containing BOTH 'load' AND 'demand'
# (matches 'Load Demand (kW)' specifically), then fall back to broader keywords.
col_lower = {col.lower(): col for col in df_raw.columns}
demand_col = None

# Priority 1: column containing both 'load' and 'demand'
for key, col in col_lower.items():
    if 'load' in key and 'demand' in key:
        demand_col = col
        break

# Priority 2: column containing 'load' or 'demand'
if demand_col is None:
    for key, col in col_lower.items():
        if ('load' in key or 'demand' in key) and col != ts_col:
            demand_col = col
            break

# Priority 3: broader energy keywords (excluding timestamp column)
if demand_col is None:
    for keyword in ['consumption', 'power', 'energy', 'kw']:
        for key, col in col_lower.items():
            if keyword in key and col != ts_col:
                demand_col = col
                break
        if demand_col:
            break

print(f"Detected timestamp column: {ts_col}")
print(f"Detected demand column: {demand_col}")

Detected timestamp column: Timestamp
Detected demand column: Load Demand (kW)


In [4]:

# Convert timestamp to datetime
df_raw[ts_col] = pd.to_datetime(df_raw[ts_col], errors='coerce')

# Sort by timestamp
df_raw = df_raw.sort_values(ts_col)

# Remove duplicate timestamps
original_len = len(df_raw)
df_raw = df_raw.drop_duplicates(subset=[ts_col], keep='first')
duplicates_removed = original_len - len(df_raw)
print(f"Removed {duplicates_removed} duplicate timestamps")

# Handle missing timestamps
df_raw = df_raw.dropna(subset=[ts_col])

print(f"Data after cleaning: {len(df_raw)} records")
print(f"Date range: {df_raw[ts_col].min()} to {df_raw[ts_col].max()}")


Removed 0 duplicate timestamps
Data after cleaning: 189793 records
Date range: 2020-01-01 00:00:00 to 2025-05-31 00:00:00


In [5]:
# Remove impossible values: missing or negative demand
before = len(df_raw)
df_clean = df_raw.dropna(subset=[demand_col])
negative_count = (df_clean[demand_col] < 0).sum()
df_clean = df_clean[df_clean[demand_col] >= 0]
print(f"Removed {before - len(df_clean)} missing or negative demand values ({negative_count} negative)")

# Remove extreme spikes above median + 10 x IQR
Q1 = df_clean[demand_col].quantile(0.25)
Q3 = df_clean[demand_col].quantile(0.75)
IQR = Q3 - Q1
spike_bound = df_clean[demand_col].median() + 10 * IQR
spikes = df_clean[demand_col] > spike_bound
df_clean = df_clean[~spikes].reset_index(drop=True)
print(f"Spike bound (median + 10 x IQR): {spike_bound:.2f} kW")
print(f"Removed {spikes.sum()} extreme spikes")

# Fill gaps using linear interpolation
df_clean[demand_col] = df_clean[demand_col].interpolate(method='linear', limit_direction='both')
df_clean[demand_col] = df_clean[demand_col].ffill().bfill()
print(f"Data after spike removal: {len(df_clean)} records")

Removed 0 missing or negative demand values (0 negative)
Spike bound (median + 10 x IQR): 5190.72 kW
Removed 0 extreme spikes
Data after spike removal: 189793 records


In [6]:
# Remove extreme outliers using the IQR method (3 x IQR, same as the app)
Q1 = df_clean[demand_col].quantile(0.25)
Q3 = df_clean[demand_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

outliers = (df_clean[demand_col] < lower_bound) | (df_clean[demand_col] > upper_bound)
outlier_count = outliers.sum()
high_count = (df_clean[demand_col] > upper_bound).sum()
df_clean = df_clean[~outliers].reset_index(drop=True)

print(f"IQR bounds: {lower_bound:.2f} to {upper_bound:.2f} kW")
print(f"Removed {outlier_count} outliers using IQR method ({high_count} above, {outlier_count - high_count} below)")

# Resample to 15-minute intervals ('15min' replaces deprecated '15T' in pandas 2.2+)
# and interpolate the readings removed above, so every day keeps 96 slots
df_indexed = df_clean.set_index(ts_col)[[demand_col]]
df_clean = df_indexed.resample('15min').mean()
df_clean[demand_col] = df_clean[demand_col].interpolate(method='linear', limit_direction='both')

print(f"After resampling to 15-minute intervals: {len(df_clean)} records")
print(f"Date range: {df_clean.index.min()} to {df_clean.index.max()}")
print(f"Final cleaned data: {len(df_clean)} records")

IQR bounds: 485.49 to 2928.49 kW
Removed 691 outliers using IQR method (691 above, 0 below)
After resampling to 15-minute intervals: 189793 records
Date range: 2020-01-01 00:00:00 to 2025-05-31 00:00:00
Final cleaned data: 189793 records


In [7]:

# Reset index to get timestamp back as column
df_clean = df_clean.reset_index()

# Add time features
df_clean['Date'] = df_clean[ts_col].dt.date
df_clean['Hour'] = df_clean[ts_col].dt.hour
df_clean['Minute'] = df_clean[ts_col].dt.minute
df_clean['DayOfWeek'] = df_clean[ts_col].dt.dayofweek
df_clean['Month'] = df_clean[ts_col].dt.month
df_clean['IsWeekend'] = (df_clean['DayOfWeek'].isin([5, 6])).astype(int)

# Reorder columns
cols = [ts_col, demand_col, 'Date', 'Hour', 'Minute', 'DayOfWeek', 'Month', 'IsWeekend']
df_clean = df_clean[cols]

print("Added time features:")
print(f"- Date: {df_clean['Date'].min()} to {df_clean['Date'].max()}")
print(f"- Hours: {df_clean['Hour'].min()} to {df_clean['Hour'].max()}")
print(f"- DayOfWeek: {sorted(df_clean['DayOfWeek'].unique())}")
print(f"- Month: {sorted(df_clean['Month'].unique())}")
print(f"- Weekend days: {df_clean['IsWeekend'].sum()}")


Added time features:
- Date: 2020-01-01 to 2025-05-31
- Hours: 0 to 23
- DayOfWeek: [0, 1, 2, 3, 4, 5, 6]
- Month: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
- Weekend days: 54145


In [8]:

# Create processed directory
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned data
df_clean.to_csv(CLEANED_DATA_PATH, index=False)

print(f"Cleaned data saved to: {CLEANED_DATA_PATH}")
print(f"Final shape: {df_clean.shape}")
print(f"File size: {CLEANED_DATA_PATH.stat().st_size / 1024 / 1024:.1f} MB")

print("\nFinal data summary:")
print(df_clean[[demand_col, 'Hour', 'DayOfWeek', 'Month']].describe())


Cleaned data saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\cleaned_data.csv
Final shape: (189793, 8)
File size: 9.3 MB

Final data summary:
       Load Demand (kW)           Hour      DayOfWeek          Month
count     189793.000000  189793.000000  189793.000000  189793.000000
mean        1743.221583      11.499939       3.000011       6.253913
std          340.108650       6.922237       1.998740       3.466477
min          629.730000       0.000000       0.000000       1.000000
25%         1531.220000       5.000000       1.000000       3.000000
50%         1699.640000      11.000000       3.000000       6.000000
75%         1879.460000      17.000000       5.000000       9.000000
max         2928.450000      23.000000       6.000000      12.000000


In [9]:
# Check against the app's cleaned data (read-only)
app_clean = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "cleaned_data.csv")

same_rows = len(app_clean) == len(df_clean)
same_timestamps = (pd.to_datetime(app_clean["Timestamp"]).values == pd.to_datetime(df_clean[ts_col]).values).all()
same_values = np.allclose(app_clean["Load Demand (kW)"].values, df_clean[demand_col].values)

print("=== CHECK AGAINST THE APP (data/processed/cleaned_data.csv) ===")
print(f"Rows: notebook {len(df_clean)} | app {len(app_clean)} | same: {same_rows}")
print(f"Same timestamps: {same_timestamps}")
print(f"Same demand values: {same_values}")

=== CHECK AGAINST THE APP (data/processed/cleaned_data.csv) ===
Rows: notebook 189793 | app 189793 | same: True
Same timestamps: True
Same demand values: True
